# BTOE End-to-End Walkthrough

**Predicting Software Maintenance Effort: An Empirical Study of Five Open-Source Repositories**

This notebook walks through the entire project from raw data to published results. It is designed to run offline against the committed replication-package artifacts — no network access, no retraining, no API calls required.

## What You'll See

1. The 4,476-pair dataset mined from GitHub
2. How text and repository features are extracted
3. The three primary models (A, B, C) and their performance
4. Visualizations (predicted vs actual, bucket grid, SHAP)
5. Robustness checks (sensitivity analysis)
6. The leave-one-repository-out (LORO) generalization test
7. Statistical comparisons (Wilcoxon + Cliff's δ)
8. The gpt-4o zero-shot baseline comparison
9. Reproducibility verification

## The Core Finding

Across every ablation, model configuration, and generalization test, Standardized Accuracy (SA) stays within a narrow **32.0–34.8%** band. The models buy only ~2.5% MAE improvement over a constant-median predictor within the training corpus, and roughly ~1% on unseen repositories. We characterize this as a **predictive ceiling** for current feature classes on this prediction target.

## Requirements

- Python 3.11+
- `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy`, `scikit-learn`, `pyarrow`, `joblib`
- Run from the repository root (`btoe/`) or the `notebooks/` folder


## 1. Setup and Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)

REPO = Path('.').resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
print(f'Repository root: {REPO}')

DATA = REPO / 'data'
MODELS = REPO / 'models'
assert DATA.exists() and MODELS.exists(), 'Run this notebook from the repo root or notebooks/ folder'


## 2. The Dataset

We mined **4,476 issue–pull-request pairs** from five large open-source repositories:

| Repository | Language | Domain | Sponsor |
|-----------|----------|--------|---------|
| microsoft/vscode | TypeScript | Editor | Microsoft |
| kubernetes/kubernetes | Go | Container orchestration | CNCF |
| rust-lang/rust | Rust | Compiler/language | Rust Foundation |
| microsoft/TypeScript | TypeScript | Language service | Microsoft |
| facebook/react | JavaScript | UI library | Meta |

Each pair links a closed issue to the merged PR that resolved it. The target variable is `duration_hours` — elapsed wall-clock time from issue creation to PR merge.


In [ ]:
raw = pd.read_parquet(DATA / 'raw' / 'issue_pr_pairs.parquet')
print(f'Total pairs: {len(raw):,}')
print(f'Repositories: {raw["repo"].nunique()}')
print()
print('Per-repository counts:')
print(raw['repo'].value_counts().to_string())


In [ ]:
raw[['repo', 'issue_number', 'issue_title', 'duration_hours']].head(3)

### 2.1 Duration Distribution

The target variable is heavily right-skewed. Most issues resolve in days; a long tail stretches into months.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(raw['duration_hours'], bins=100, log=True, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Duration (hours)')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Raw duration distribution')
axes[0].axvline(raw['duration_hours'].median(), color='red', linestyle='--',
                label=f'median: {raw["duration_hours"].median():.0f}h')
axes[0].axvline(raw['duration_hours'].mean(), color='orange', linestyle='--',
                label=f'mean: {raw["duration_hours"].mean():.0f}h')
axes[0].legend()

axes[1].hist(np.log1p(raw['duration_hours']), bins=50, color='steelblue', edgecolor='black')
axes[1].set_xlabel('log(1 + duration_hours)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log-transformed target (what the model learns)')

plt.tight_layout()
plt.show()

print(f'Range: {raw["duration_hours"].min():.1f}h to {raw["duration_hours"].max():.1f}h')
print(f'Mean: {raw["duration_hours"].mean():.1f}h, Median: {raw["duration_hours"].median():.1f}h')
print(f'Std:  {raw["duration_hours"].std():.1f}h')


The paper applies a primary filter of **1 hour to 90 days** (2,160 hours). This removes outliers that likely represent accidental issue closures (<1h) or abandoned issues (>90d), leaving **4,321 pairs** for training and evaluation.


In [ ]:
filtered = raw[(raw['duration_hours'] >= 1) & (raw['duration_hours'] <= 2160)].copy()
print(f'After duration filter: {len(filtered):,} pairs (removed {len(raw) - len(filtered)})')
print(f'Train/test split (80/20): {int(len(filtered) * 0.8):,} / {int(len(filtered) * 0.2):,}')


## 3. Feature Extraction

Two parallel feature families are extracted per issue–PR pair.

### 3.1 Text Features (Model A)

- **768-dim CodeBERT embedding** of concatenated issue title + body (no PR text — prevents data leakage)
- Reduced to **50 PCA components** fit on training data only
- Plus **6 derived NLP features**: complexity_score, cross_ref_density, ambiguity_index, technical_scope, text_length, word_count

Total: **56 features**


In [ ]:
nlp = pd.read_parquet(DATA / 'processed' / 'nlp_features.parquet')
print(f'Rows: {len(nlp):,}')
print(f'Columns: {len(nlp.columns)} ({sum(c.startswith("emb_") for c in nlp.columns)} embeddings + 6 derived)')
print()
derived_cols = [c for c in nlp.columns if not c.startswith('emb_')]
print('Derived feature statistics:')
nlp[derived_cols].describe().round(2)


### 3.2 Repository Features (Model B)

13 structural metrics extracted via PyDriller from the resolving PR's commit history:
- Churn: lines added, deleted, commit count, average per file
- Coupling: fan-in, fan-out, change-impact radius
- File maturity: average/min/max file age (days)
- Change frequency: average/max per file
- `num_files` — number of files touched (post-hoc signal, excluded in Model B-restricted)


In [ ]:
repo_feat = pd.read_parquet(DATA / 'processed' / 'repo_features.parquet')
print(f'Rows: {len(repo_feat):,}')
print(f'Features: {list(repo_feat.columns)}')
print()
repo_feat.describe().round(2)


### 3.3 Model C = Combined

Model C concatenates Model A's 56 features with Model B's 13 features = **69 features**. Same training protocol.


## 4. Primary Results (Table II)

All three models were trained with:
- XGBoost regressor with `objective="reg:squarederror"`
- **Bayesian hyperparameter optimization**: 50 iterations, 5-fold CV on training set
- Log-transformed target: `np.log1p(duration_hours)`
- Fixed random seed (42) for reproducibility
- 80/20 train/test split

**Standardized Accuracy (SA)** follows Shepperd & MacDonell (2012):

$$SA = \left(1 - \frac{MAE_{model}}{MAE_{P_0}}\right) \times 100$$

where $MAE_{P_0}$ is the mean MAE across **1,000 random guessing runs** sampled with replacement from the training distribution.


In [ ]:
with open(MODELS / 'results.json') as f:
    results = json.load(f)

# results.json has flat structure: {'model_a': {metrics}, 'model_b': {metrics}, ...}
rows = []
for key, m in results.items():
    if not isinstance(m, dict) or 'mae' not in m:
        continue
    rows.append({
        'Model': key.replace('_', ' ').title(),
        'MAE (h)': round(m['mae'], 1),
        'MdAE (h)': round(m['mdae'], 1),
        'MMRE': round(m['mre'], 2),
        'PRED(25) %': round(m['pred_25'], 1),
        'PRED(50) %': round(m['pred_50'], 1),
        'R²': round(m['r2'], 3),
        'SA %': round(m['sa'], 2),
        'MAE_P0 (h)': round(m['mae_p0'], 1),
        'Glass Δ': round(m['glass_delta'], 2),
    })

primary = pd.DataFrame(rows)
primary


### 4.1 How to Read These Numbers

- **MAE 326.3h for Model A**: on average, our prediction is off by ~2 weeks of calendar time
- **SA 34.3%**: Model A is 34.3% better than random guessing from the training distribution
- **Glass's Δ = 11.53**: Model A is 11.5 standard deviations better than random (anything > 0.8 is "large")
- **R² negative**: on squared-error terms, predicting the training mean beats all three models — a known pattern for heavy-tailed distributions
- **PRED(25) = 8.1%**: only 8% of predictions are within 25% of the actual value
- **PRED(50) = 19.5%**: fewer than one in five predictions is within 50%

The gap between high SA (34%) and low PRED(25) (8%) reveals the tension: the models beat random guessing meaningfully, but are still far from precise.


## 5. Visualizations

### 5.1 Predicted vs Actual (Model A)

The regression-to-the-mean pattern is unmistakable: predictions cluster between 100h and 500h, while actuals span 1h to 2000h+.


In [ ]:
preds_a = pd.read_csv(MODELS / 'model_a_text_only' / 'predictions.csv')
test_preds = preds_a[preds_a['split'] == 'test'].copy()

fig, ax = plt.subplots(figsize=(10, 10))

max_val = max(test_preds['actual_hours'].max(), test_preds['predicted_hours'].max())
ax.plot([1, max_val], [1, max_val], 'r--', alpha=0.5, label='Perfect prediction')

x = np.linspace(1, max_val, 100)
ax.fill_between(x, x * 0.75, x * 1.25, alpha=0.1, color='green', label='±25% band')

ax.scatter(test_preds['actual_hours'], test_preds['predicted_hours'],
           alpha=0.4, s=20, color='steelblue')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Actual duration (hours, log scale)')
ax.set_ylabel('Predicted duration (hours, log scale)')
ax.set_title(f'Model A: Predicted vs Actual (n={len(test_preds)})')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'Actual range:    {test_preds["actual_hours"].min():.1f}h to {test_preds["actual_hours"].max():.1f}h')
print(f'Predicted range: {test_preds["predicted_hours"].min():.1f}h to {test_preds["predicted_hours"].max():.1f}h')


### 5.2 Bucket Grid

How often does the model predict the correct duration *bucket*? The diagonal shows correct bucket predictions; off-diagonal shows which buckets get confused.


In [ ]:
def to_bucket(hours):
    if hours < 24: return '<1 day'
    if hours < 72: return '1-3 days'
    if hours < 168: return '3-7 days'
    if hours < 672: return '1-4 weeks'
    return '>4 weeks'

test_preds['actual_bucket'] = test_preds['actual_hours'].apply(to_bucket)
test_preds['pred_bucket'] = test_preds['predicted_hours'].apply(to_bucket)

order = ['<1 day', '1-3 days', '3-7 days', '1-4 weeks', '>4 weeks']
grid = pd.crosstab(test_preds['actual_bucket'], test_preds['pred_bucket'])
grid = grid.reindex(index=order, columns=order, fill_value=0)
grid_pct = grid.div(grid.sum(axis=1), axis=0).fillna(0) * 100

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(grid_pct, annot=True, fmt='.0f', cmap='Blues',
            cbar_kws={'label': '% of actuals in row'}, ax=ax)
ax.set_xlabel('Predicted bucket')
ax.set_ylabel('Actual bucket')
ax.set_title('Model A: Bucket-level agreement (row-normalized %)')
plt.tight_layout()
plt.show()

correct = np.sum(np.diag(grid))
total = grid.values.sum()
print(f'Correct bucket predictions: {correct}/{total} ({100*correct/total:.1f}%)')


### 5.3 SHAP Feature Importance

SHAP (SHapley Additive exPlanations) values quantify each feature's average contribution to predictions.


In [ ]:
for model_name in ['model_a_text_only', 'model_b_repo_only', 'model_c_combined']:
    shap_file = MODELS / model_name / 'analysis' / 'shap_values.csv'
    if shap_file.exists():
        shap_df = pd.read_csv(shap_file).sort_values('mean_abs_shap', ascending=False).head(5)
        print(f'\n{model_name} — Top 5 features:')
        for _, row in shap_df.iterrows():
            print(f'  {row["feature"]:30s} {row["mean_abs_shap"]:.4f}')
    else:
        print(f'{model_name}: no SHAP file')


## 6. Sensitivity Analysis (Table V)

Does the ceiling depend on where we cap duration? We retrained all three models at five thresholds: 14, 30, 60, 90 days, and no cap.


In [ ]:
sens = pd.read_csv(MODELS / 'sensitivity_analysis.csv')
sens


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

threshold_order = ['14d', '30d', '60d', '90d', 'no cap']
sens_sorted = sens.set_index('threshold').reindex(threshold_order).reset_index()
x_pos = np.arange(len(threshold_order))

for model, col in [('A', 'a_sa'), ('B', 'b_sa'), ('C', 'c_sa')]:
    if col in sens_sorted.columns:
        ax.plot(x_pos, sens_sorted[col], marker='o', linewidth=2, markersize=10, label=f'Model {model}')

ax.set_xticks(x_pos)
ax.set_xticklabels(threshold_order)
ax.set_xlabel('Duration filter threshold')
ax.set_ylabel('Standardized Accuracy (SA %)')
ax.set_title('SA across duration thresholds (higher is better)')
ax.legend()
plt.tight_layout()
plt.show()


**Key observation:** SA rises with wider thresholds, but this is an *artifact of the random baseline*. As the threshold expands, the random baseline MAE grows faster than the model MAE because random sampling from a heavier-tailed distribution incurs disproportionately large errors.

The practically meaningful claim is that **Model C never significantly outperforms Model A at any threshold** (p > 0.20 at all 5).


## 7. Leave-One-Repository-Out (Table VIII)

The pooled 80/20 split mixes issues from all 5 repositories into both training and test sets. This gives optimistic generalization numbers. **LORO** holds out one entire repository, trains on the other four, and evaluates on the held-out one — a much cleaner test of cross-repository generalization.


In [ ]:
loro = pd.read_csv(MODELS / 'loro_results.csv')
display_cols = ['held_out_repo', 'n_test', 'mae_A', 'mae_C', 'median_predictor_mae',
                'sa_A', 'sa_C', 'wilcoxon_A_vs_C_p']
loro[display_cols].round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

repos = loro['held_out_repo'].str.split('/').str[1]
x = np.arange(len(repos))
width = 0.35

ax.bar(x - width/2, loro['mae_A'], width, label='Model A', color='steelblue')
ax.bar(x + width/2, loro['median_predictor_mae'], width,
       label='Constant-median predictor', color='orange')

ax.set_xticks(x)
ax.set_xticklabels(repos, rotation=15, ha='right')
ax.set_ylabel('MAE (hours)')
ax.set_title('Model A vs constant-median predictor on unseen repositories (LORO)')
ax.legend()
plt.tight_layout()
plt.show()

mean_sa_a = loro['sa_A'].mean()
std_sa_a = loro['sa_A'].std()
gain = (loro['median_predictor_mae'] - loro['mae_A']).mean()
wins = (loro['mae_A'] < loro['median_predictor_mae']).sum()
print(f'\nModel A mean SA across folds: {mean_sa_a:.2f} (SD {std_sa_a:.2f})')
print(f'Model A SA range: {loro["sa_A"].min():.1f}% to {loro["sa_A"].max():.1f}%')
print(f'Mean gain over median predictor: {gain:.1f}h')
print(f'Folds where model beats median: {wins}/5')


**The core LORO finding:** Model A beats the constant-median predictor by only **4.5 hours on average**, and that gain is driven almost entirely by one fold (vscode, +22h). On the other 4 repositories, the gain is under 1 hour. On kubernetes, Model A is **0.8 hours worse** than the median predictor.

> *When generalization is measured honestly, the learned models match the median predictor rather than beat it.*


## 8. Statistical Comparisons

We use the **Wilcoxon signed-rank test** (paired, non-parametric) for significance and **Cliff's δ** for effect size. With n=865 test samples, Wilcoxon can detect statistically significant differences that have negligible practical effect — so effect size matters.

Cliff's δ interpretation (Romano et al., 2006):
- |δ| < 0.147 → negligible
- 0.147 ≤ |δ| < 0.33 → small
- 0.33 ≤ |δ| < 0.474 → medium
- |δ| ≥ 0.474 → large


In [ ]:
preds_a = pd.read_csv(MODELS / 'model_a_text_only' / 'predictions.csv')
preds_b = pd.read_csv(MODELS / 'model_b_repo_only' / 'predictions.csv')
preds_c = pd.read_csv(MODELS / 'model_c_combined' / 'predictions.csv')

def errors(df):
    test = df[df['split'] == 'test']
    return np.abs(test['actual_hours'].values - test['predicted_hours'].values)

def cliffs_delta(x, y):
    n = len(x)
    more = int(np.sum(x[:, None] > y[None, :]))
    less = int(np.sum(x[:, None] < y[None, :]))
    return (more - less) / (n * n)

def interpret(d):
    if abs(d) < 0.147: return 'negligible'
    if abs(d) < 0.33: return 'small'
    if abs(d) < 0.474: return 'medium'
    return 'large'

e_a, e_b, e_c = errors(preds_a), errors(preds_b), errors(preds_c)

rows = []
for label, x, y in [('A vs B', e_a, e_b), ('A vs C', e_a, e_c), ('B vs C', e_b, e_c)]:
    stat, p = stats.wilcoxon(x, y)
    delta = cliffs_delta(x, y)
    rows.append({
        'Comparison': label,
        'Wilcoxon W': int(stat),
        'p-value': round(p, 4),
        'Significant (α=0.05)': 'Yes' if p < 0.05 else 'No',
        "Cliff's δ": round(delta, 4),
        'Effect size': interpret(delta),
    })

pd.DataFrame(rows)


**Result:** All Cliff's δ values are well below the negligibility threshold (0.147). Even where differences are statistically significant (A vs B, B vs C), the effect sizes are tiny. The A-vs-C comparison — the key test of whether combining features helps — is not significant and has negligible effect size.


## 9. gpt-4o Zero-Shot Baseline Comparison

Reviewer B asked why we used CodeBERT/XGBoost rather than a frontier LLM. We ran gpt-4o as a zero-shot baseline on the same 865 test issues (temperature 0, max 200 output tokens).


In [ ]:
llm_results_path = MODELS / 'llm_baseline' / 'gpt-4o' / 'results.json'
if llm_results_path.exists():
    with open(llm_results_path) as f:
        llm_results = json.load(f)
    tm = llm_results.get('test_metrics', llm_results)

    print('gpt-4o zero-shot on 865 test issues:')
    for key, label in [('mae', 'MAE'), ('mdae', 'MdAE'), ('pred_25', 'PRED(25)'),
                        ('sa', 'SA'), ('mae_p0', 'MAE_P0'), ('glass_delta', 'Glass Δ')]:
        if key in tm:
            unit = 'h' if 'ae' in key or key == 'mae_p0' else ('%' if key in ('sa', 'pred_25') else '')
            print(f'  {label:12s} {tm[key]:>8.2f}{unit}')
    print()
    print('Compared to Model A (text-only XGBoost): SA 34.3%, MAE 326.3h')
    print('The frontier LLM trails our tuned pipeline — reinforces the ceiling claim.')
else:
    print('gpt-4o results not present on this branch.')
    print('They live on experiment/llm-baseline; checkout that branch to inspect them.')


## 10. Live Prediction Demo

Load the saved Model A and inspect its hyperparameters. This requires the `.joblib` files, which are present only when models have been trained locally (they are excluded from the replication-package).


In [ ]:
import joblib

model_dir = MODELS / 'model_a_text_only'
required = ['model.joblib', 'pca.joblib', 'scaler.joblib']
have_models = all((model_dir / f).exists() for f in required)

if not have_models:
    print('Trained model binaries not present (expected on replication-package branch).')
    print('To run live predictions, train Models first:')
    print('  python -m src.pipeline --step train')
    print()
    print('To experiment interactively, launch the Flask web app:')
    print('  python app.py')
else:
    print('Loading Model A...')
    model = joblib.load(model_dir / 'model.joblib')
    with open(model_dir / 'results.json') as f:
        r = json.load(f)
    print(f'XGBoost best params (found via Bayesian optimization):')
    for k, v in r.get('best_params', {}).items():
        if isinstance(v, float):
            print(f'  {k:20s} {v:.4g}')
        else:
            print(f'  {k:20s} {v}')
    print()
    print('For an end-to-end prediction on new text, use the Flask web app:')
    print('  python app.py')


## 11. Reproducibility Verification

The committed `predictions.csv` for each model should reproduce the metrics in `results.json` exactly. Let's verify.


In [ ]:
for model_name in ['model_a_text_only', 'model_b_repo_only', 'model_c_combined']:
    with open(MODELS / model_name / 'results.json') as f:
        results = json.load(f)

    preds = pd.read_csv(MODELS / model_name / 'predictions.csv')
    test = preds[preds['split'] == 'test']

    actual = test['actual_hours'].values
    predicted = test['predicted_hours'].values
    errors = np.abs(actual - predicted)

    tm = results.get('test_metrics', {})
    recomputed_mae = errors.mean()
    recomputed_mdae = np.median(errors)
    stored_mae = tm.get('mae', 0)
    stored_mdae = tm.get('mdae', 0)

    mae_ok = abs(recomputed_mae - stored_mae) < 0.1
    mdae_ok = abs(recomputed_mdae - stored_mdae) < 0.1

    status = '✓' if (mae_ok and mdae_ok) else '✗'
    print(f'{status} {model_name}: MAE {recomputed_mae:.2f} (stored {stored_mae:.2f}), '
          f'MdAE {recomputed_mdae:.2f} (stored {stored_mdae:.2f})')


## 12. What's Next?

This notebook covered the committed artifacts. To actually re-run the pipeline:

```bash
# 1. Collect data (requires GitHub API token, ~4-6 hours)
python -m src.pipeline --step collect

# 2. Extract features (CodeBERT ~30 min on CPU, PyDriller ~2-4 hours)
python -m src.pipeline --step nlp_features
python -m src.pipeline --step repo_features

# 3. Train (Bayesian hyperparameter search ~30 min)
python -m src.pipeline --step train

# 4. Analyses
python -m src.pipeline --step analyze
python -m src.pipeline --step shap_analysis
python -m src.pipeline --step error_analysis
python -m src.pipeline --step sensitivity
python -m src.pipeline --step encoder_ablation
python -m src.pipeline --step feature_selection_ablation
python -m src.experiments.loro_eval
```

See `REPLICATION.md` for the full step-by-step replication guide including offline verification.

---

**Paper:** *Predicting Software Maintenance Effort: An Empirical Study of Five Open-Source Repositories* (SANER 2027 RENE submission)
